# Granite Peak CRE: 08 App permissions

Grants the Databricks App's service principal what it needs, then checks each grant:

- **Unity Catalog:** `USE CATALOG`, `USE SCHEMA`, and `READ VOLUME` on `landing` (to serve lease PDFs).
- **Lakebase Postgres:** `SELECT` on the synced tables and read/write on `app.abstract_review`. A Unity Catalog grant on a synced table does **not** authorize Postgres reads, so these are granted directly to the app's Postgres role.

Genie runs on behalf of the signed-in user, so the app service principal needs no access to the row-filtered gold tables.

In [1]:
%pip install -q "psycopg[binary]>=3.1" "databricks-sdk>=0.60"
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
dbutils.widgets.text("catalog", "smmorgan_catalog", "Catalog")
dbutils.widgets.text("schema", "granite_commercial_real_estate", "Schema")
dbutils.widgets.text("app_name", "granite-cre-portfolio", "App")
dbutils.widgets.text("instance", "granite-cre", "Lakebase instance")
catalog, schema = dbutils.widgets.get("catalog").strip(), dbutils.widgets.get("schema").strip()
APP, INSTANCE = dbutils.widgets.get("app_name").strip(), dbutils.widgets.get("instance").strip()

import uuid, psycopg
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
sp = w.apps.get(APP).service_principal_client_id
print(f"app {APP} runs as service principal {sp}")

app granite-cre-portfolio runs as service principal dc695a23-3092-4b40-b6e0-961733b766e9


In [3]:
for stmt in (f"GRANT USE CATALOG ON CATALOG `{catalog}` TO `{sp}`",
             f"GRANT USE SCHEMA ON SCHEMA `{catalog}`.`{schema}` TO `{sp}`",
             f"GRANT READ VOLUME ON VOLUME `{catalog}`.`{schema}`.`landing` TO `{sp}`"):
    try:
        spark.sql(stmt); print("ok  ", stmt)
    except Exception as e:
        print("FAIL", stmt, "->", str(e).splitlines()[0][:160])

ok   GRANT USE CATALOG ON CATALOG `smmorgan_catalog` TO `dc695a23-3092-4b40-b6e0-961733b766e9`
ok   GRANT USE SCHEMA ON SCHEMA `smmorgan_catalog`.`granite_commercial_real_estate` TO `dc695a23-3092-4b40-b6e0-961733b766e9`
ok   GRANT READ VOLUME ON VOLUME `smmorgan_catalog`.`granite_commercial_real_estate`.`landing` TO `dc695a23-3092-4b40-b6e0-961733b766e9`


In [4]:
inst = w.database.get_database_instance(INSTANCE)
cred = w.database.generate_database_credential(request_id=str(uuid.uuid4()), instance_names=[INSTANCE])
with psycopg.connect(host=inst.read_write_dns, dbname="databricks_postgres", user=w.current_user.me().user_name,
                     password=cred.token, sslmode="require", autocommit=True) as conn:
    for stmt in (f'GRANT USAGE ON SCHEMA "{schema}" TO "{sp}"',
                 f'GRANT SELECT ON ALL TABLES IN SCHEMA "{schema}" TO "{sp}"',
                 f'ALTER DEFAULT PRIVILEGES IN SCHEMA "{schema}" GRANT SELECT ON TABLES TO "{sp}"',
                 f'GRANT USAGE ON SCHEMA app TO "{sp}"',
                 f'GRANT SELECT, INSERT, UPDATE ON app.abstract_review TO "{sp}"'):
        conn.execute(stmt); print("ok  ", stmt)
    rows = conn.execute("""
        SELECT table_schema, table_name, string_agg(privilege_type, ', ' ORDER BY privilege_type)
        FROM information_schema.role_table_grants WHERE grantee = %s GROUP BY 1, 2 ORDER BY 1, 2""", (sp,)).fetchall()
print("Postgres privileges held by the app:")
for r in rows:
    print("  ", r)

ok   GRANT USAGE ON SCHEMA "granite_commercial_real_estate" TO "dc695a23-3092-4b40-b6e0-961733b766e9"
ok   GRANT SELECT ON ALL TABLES IN SCHEMA "granite_commercial_real_estate" TO "dc695a23-3092-4b40-b6e0-961733b766e9"
ok   ALTER DEFAULT PRIVILEGES IN SCHEMA "granite_commercial_real_estate" GRANT SELECT ON TABLES TO "dc695a23-3092-4b40-b6e0-961733b766e9"
ok   GRANT USAGE ON SCHEMA app TO "dc695a23-3092-4b40-b6e0-961733b766e9"
ok   GRANT SELECT, INSERT, UPDATE ON app.abstract_review TO "dc695a23-3092-4b40-b6e0-961733b766e9"
Postgres privileges held by the app:
   ('app', 'abstract_review', 'INSERT, SELECT, UPDATE')
   ('granite_commercial_real_estate', 'lease_abstract_synced', 'SELECT')
   ('granite_commercial_real_estate', 'lease_discrepancies_synced', 'SELECT')
   ('granite_commercial_real_estate', 'property_360_synced', 'SELECT')
   ('granite_commercial_real_estate', 'renewal_scores_synced', 'SELECT')
